
# Insurance Lakehouse with DuckDB
Bronze → Silver → Gold architecture example.


In [1]:

import duckdb
from pathlib import Path

BASE_DIR = Path(".")
BRONZE_DIR = BASE_DIR / "insurance" / "data" / "bronze"
print("BRONZE_DIR", BRONZE_DIR)


BRONZE_DIR insurance/data/bronze


In [2]:
con = duckdb.connect("insurance_lakehouse.duckdb")

## Bronze Layer

In [5]:

con.execute("""
DROP VIEW IF EXISTS bronze_insurance;

CREATE VIEW bronze_insurance AS
SELECT
    *,
    CAST(split_part(filename, '.', 2) AS INTEGER) AS year
FROM read_csv_auto(
    'insurance/data/bronze/insurance.*.csv',
    header = true,
    filename = true
);
""")

con.execute("SELECT year, COUNT(*) as rows FROM bronze_insurance GROUP BY year ORDER BY year").df()


,year,rows
0,2021,1300
1,2022,1300
2,2023,1300
3,2024,1300


## Silver Layer

In [6]:

con.execute("""
DROP TABLE IF EXISTS silver_insurance;

CREATE TABLE silver_insurance AS
SELECT
    age::INTEGER AS age,
    gender,
    bmi::DOUBLE AS bmi,
    children::INTEGER AS children,
    region,
    charges::DOUBLE AS charges,
    year::INTEGER AS year
FROM bronze_insurance
""")

con.execute("SELECT year, COUNT(*) FROM silver_insurance GROUP BY year ORDER BY year").df()


,year,count_star()
0,2021,1300
1,2022,1300
2,2023,1300
3,2024,1300


## Gold Layer

In [7]:

con.execute("""
DROP VIEW IF EXISTS gold_kpi_year;

CREATE VIEW gold_kpi_year AS
SELECT
    year,
    COUNT(*) members,
    AVG(charges) avg_charges,
    SUM(charges) total_charges
FROM silver_insurance
GROUP BY year
ORDER BY year
""")

con.execute("SELECT * FROM gold_kpi_year").df()


,year,members,avg_charges,total_charges
0,2021,1300,15748.940800,20473623.04
1,2022,1300,16781.303192,21815694.15
2,2023,1300,17149.925223,22294902.79
3,2024,1300,17726.495423,23044444.05


## Example Insights

In [8]:
con.execute("""SELECT year, AVG(charges) avg_charges FROM silver_insurance GROUP BY year ORDER BY year""").df()

,year,avg_charges
0,2021,15748.940800
1,2022,16781.303192
2,2023,17149.925223
3,2024,17726.495423


In [9]:
con.execute("""SELECT region, AVG(charges) avg_charges FROM silver_insurance GROUP BY region""").df()

,region,avg_charges
0,southwest,16417.575123
1,southeast,16756.908670
2,northwest,16997.721412
3,northeast,17247.142607


In [10]:
con.execute("""SELECT smoker, AVG(charges) avg_charges FROM bronze_insurance GROUP BY smoker""").df()

,smoker,avg_charges
0,False,12191.119137
1,True,37149.764367


In [11]:
con.execute("""SELECT age, charges FROM silver_insurance ORDER BY charges DESC LIMIT 10""").df()

,age,charges
0,62,61974.25
1,55,61573.60
2,55,61413.30
3,55,59509.57
4,52,59370.34
5,61,59349.88
6,42,56564.29
7,57,56559.54
8,46,56449.88
9,34,55569.29
